# AELIONIX BLACKFORGE — Phase 7 Colab Validation

This notebook performs a deterministic, one-click validation of the Blackforge **Authentication & Authorization Security Capability Foundation** (Phase 7).

**What this validates:**
- Repository integrity and commit verification
- Dependency installation (runtime + dev extras)
- All Blackforge imports, including the new `blackforge.auth` modules
- Full automated test suite (auth phase included)
- Bootstrap + health verification, including the new `auth_ready` check (11 typed auth capabilities)
- Eleven typed authentication/authorization capabilities (authentication-surface, session, scheme detection, OAuth metadata, OIDC metadata, MFA surface, authorization surface, role, permission, resource-access, access-control comparison)
- The full auth pipeline: capability -> mock transport -> normalization -> evidence (artifact + typed observations, `DERIVED_FROM`) -> World Model -> best-effort memory link
- **Observation-only by design**: no credential submission, guessing, forgery, escalation, or brute force is possible through this surface
- **Redaction at the boundary**: session values, tokens, and credential values are persisted only as one-way digests or the literal `REDACTED` marker
- Authorized, explicit **test identities** are required for access-validation capabilities; missing identities are rejected (never guessed)
- Authorization enforced *before* any transport execution (out-of-scope targets are denied); unknown capabilities are rejected
- Failure-aware statuses (RATE_LIMITED, REQUEST_FAILED, NO_EVIDENCE, LIMITED, PARTIAL, SUCCESS)
- Confidence policy: PASSIVE→LOW; direct ACTIVE kinds→HIGH; observed document kinds→MEDIUM; validated resource-access/access-control→HIGH
- World Model semantics: APPLICATION named by hostname; AUTHENTICATION entity named by scheme (namespaced by host); ENDPOINT --REQUIRES--> AUTHENTICATION; IDENTITY --HAS_ROLE--> ROLE --HAS_PERMISSION--> PERMISSION --APPLIES_TO--> RESOURCE; analysis assertions bound to the host APPLICATION or IDENTITY (validated status for exercised access)


---


In [ ]:
import sys
import platform

print("Blackforge Phase 7 Colab Validation (Authentication & Authorization Capability Foundation)")
print("=" * 60)
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("Architecture:", platform.machine())
print("=" * 60)

assert sys.version_info >= (3, 10), f"Blackforge requires Python 3.10+, got {sys.version}"
print("Python version check: PASS")


---


In [ ]:
from pathlib import Path
import subprocess
import sys
import os
import shutil

# -- Configuration (edit here if fork changes) ---------------------------
REPO_URL = "https://github.com/Sagelord00000001/Blackforge.git"
REPO_DIR = Path("/content/blackforge")
# -----------------------------------------------------------------------

if REPO_DIR.exists() and (REPO_DIR / "blackforge" / "__init__.py").exists():
    print(f"Repository already exists at {REPO_DIR}, updating...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(str(REPO_DIR))
print(f"Repository ready at {REPO_DIR}")


---


In [ ]:
import subprocess

try:
    commit = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    print("Commit:", commit)
except Exception as e:
    print("Commit unavailable (expected in scratch checkouts):", e)


---


In [ ]:
!pip install hatchling --quiet
!pip install -e ".[dev]" --quiet


---


In [ ]:
import importlib

modules = [
    "blackforge",
    "blackforge.core.config",
    "blackforge.core.errors",
    "blackforge.core.types",
    "blackforge.runtime.bootstrap",
    "blackforge.memory",
    "blackforge.evidence",
    "blackforge.world_model",
    "blackforge.world_model.models",
    "blackforge.world_model.canonical",
    "blackforge.world_model.rules",
    "blackforge.world_model.query",
    "blackforge.world_model.repository",
    "blackforge.world_model.store",
    "blackforge.world_model.materializer",
    "blackforge.mission.manager",
    "blackforge.capabilities.registry",
    "blackforge.authorization",
    "blackforge.scope.models",
    "blackforge.scope.validator",
    "blackforge.recon",
    "blackforge.recon.models",
    "blackforge.recon.mock",
    "blackforge.recon.normalization",
    "blackforge.recon.evidence",
    "blackforge.recon.materializer",
    "blackforge.recon.capabilities",
    "blackforge.recon.engine",
    "blackforge.webapi",
    "blackforge.webapi.models",
    "blackforge.webapi.mock",
    "blackforge.webapi.normalization",
    "blackforge.webapi.evidence",
    "blackforge.webapi.capabilities",
    "blackforge.webapi.engine",
    "blackforge.webapi.materializer",
    "blackforge.webapi.redaction",
    "blackforge.auth",
    "blackforge.auth.models",
    "blackforge.auth.redaction",
    "blackforge.auth.transport",
    "blackforge.auth.normalization",
    "blackforge.auth.evidence",
    "blackforge.auth.capabilities",
    "blackforge.auth.materializer",
    "blackforge.auth.engine",
]

_import_failures = []
for module in modules:
    try:
        importlib.import_module(module)
    except Exception as e:
        _import_failures.append((module, str(e)))

if _import_failures:
    for mod, err in _import_failures:
        print(f"  FAIL: {mod} — {err}")
    raise RuntimeError(f"Import health check failed: {len(_import_failures)} module(s)")

print(f"Blackforge imports OK ({len(modules)} modules verified).")
print("Auth module imports: PASS")


---


In [ ]:
import subprocess
import sys

print("Running automated test suite...")
# The LLM/torch-heavy files are excluded: importing the HF provider pulls
# ~2GB of torch memory and can SIGKILL the kernel on CPU runtimes. Those
# tests are validated locally and in the Phase 1 notebook.
result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q", "--tb=short",
        "--ignore=tests/test_huggingface_provider.py",
        "--ignore=tests/test_loader.py",
        "--ignore=tests/test_smoke_real_model.py",
    ],
    capture_output=True, text=True, cwd=str(REPO_DIR),
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:] if result.stderr else "")
    raise RuntimeError(f"pytest failed with exit code {result.returncode}")

print("Automated test suite: PASS")


---


In [ ]:
import os
from pathlib import Path

DBROOT = Path("data/phase7_colab").resolve()
DBROOT.mkdir(parents=True, exist_ok=True)
os.environ["BLACKFORGE_DB_PATH"] = str(DBROOT / "blackforge.db")
os.environ["BLACKFORGE_MEMORY_DB_PATH"] = str(DBROOT / "memory.db")
os.environ["BLACKFORGE_EVIDENCE_DB_PATH"] = str(DBROOT / "evidence.db")
os.environ["BLACKFORGE_WORLD_MODEL_DB_PATH"] = str(DBROOT / "world_model.db")
for _p in (DBROOT / "evidence.db", DBROOT / "world_model.db"):
    _p.unlink(missing_ok=True)

from blackforge.runtime.bootstrap import bootstrap

app = bootstrap()
assert app.healthy(), "Blackforge health check failed"
verification = app.verify()
for key in ("evidence_store_ready", "evidence_memory_link_ready", "memory_ready",
            "world_model_ready", "recon_ready", "webapi_ready", "auth_ready"):
    assert verification[key], f"{key} must be True"
assert verification["auth_ready"] is True, "auth_ready must be True (11 typed capabilities)"

BOOTSTRAP_OK = app.healthy() and bool(verification["auth_ready"])

for k, v in verification.items():
    symbol = "PASS" if v else "FAIL"
    print(f"  [{symbol}] {k}")

print("\nBlackforge bootstrap (auth ready, 11 capabilities): PASS")


---


In [ ]:
from blackforge.auth.models import AuthRequest
from blackforge.scope.models import TargetScope, Target, detect_target_type
from blackforge.core.types import RiskLevel

def _target(value: str) -> Target:
    return Target(value=value, target_type=detect_target_type(value))

MID = "mission_phase7_auth"
DEMO_HOSTS = [
    "web.example.com", "api.example.com", "auth.example.com",
    "legacy.example.com", "mail.example.com",
    "throttled.example.com", "unreachable.example.com",
]
scope = TargetScope(
    mission_id=MID,
    allowed_targets=[_target(t) for t in DEMO_HOSTS],
    max_risk_level=RiskLevel.HIGH,
)
req = AuthRequest(
    mission_id=MID,
    scope=scope,
    test_identities=["alice", "bob"],
    max_observations=2000,
)

engine = app.auth_engine
assert engine is not None and len(engine.capabilities) == 11
expected = sorted([
    "auth.authentication_surface",
    "auth.session_observation",
    "auth.authentication_scheme_detection",
    "auth.oauth_metadata_observation",
    "auth.oidc_metadata_observation",
    "auth.mfa_surface_observation",
    "auth.authorization_surface",
    "auth.role_observation",
    "auth.permission_observation",
    "auth.resource_access_observation",
    "auth.access_control_comparison",
])
ids_seen = sorted(c.capability_id for c in engine.capabilities)
assert ids_seen == expected, (ids_seen, expected)
print("Registered auth capabilities:", ", ".join(c.capability_id for c in engine.capabilities))

_meta_by_id = {c.capability_id: c.meta() for c in engine.capabilities}
for capability_id in expected:
    meta = _meta_by_id[capability_id]
    print(
        f"  {meta.id:<35} risk={meta.risk_level.value:<6} mode={meta.mode.value:<7} "
        f"targets={[t.value for t in meta.supported_target_types]}"
    )
CAPS_OK = True

res = engine.observe_authentication_surface(req, "web.example.com")
assert res.status.value == "success"
assert res.capability_id == "auth.authentication_surface"
print(
    f"\nauthentication_surface -> {res.observations[0].host} : "
    f"{res.observations[0].scheme} [{res.observations[0].scheme_type}]"
)


---


In [ ]:
from blackforge.evidence.models import EvidenceRelation
from blackforge.world_model.query import RelationshipQuery
from blackforge.world_model.models import EntityType, RelationshipType

r_surface = engine.observe_authentication_surface(req, "web.example.com")
r_session = engine.observe_session_details(req, "web.example.com")
r_scheme = engine.detect_authentication_schemes(req, "api.example.com")
r_oauth = engine.observe_oauth_metadata(req, "api.example.com")
r_oidc = engine.observe_oidc_metadata(req, "api.example.com")
r_mfa = engine.observe_mfa_surface(req, "web.example.com")
r_authz = engine.observe_authorization_surface(req, "web.example.com")
r_roles = engine.observe_roles(req, "web.example.com")
r_perm = engine.observe_permissions(req, "web.example.com")
r_access = engine.observe_resource_access(req, "web.example.com")
r_cmp = engine.compare_access_control(req, "web.example.com")

pipeline_runs = (r_surface, r_session, r_scheme, r_oauth, r_oidc, r_mfa,
                 r_authz, r_roles, r_perm, r_access, r_cmp)
kinds = sorted({o.kind for r in pipeline_runs for o in r.observations})
print("Pipeline observation kinds observed:", ", ".join(kinds))
for expected_kind in ("auth_surface", "session", "auth_scheme", "oauth_metadata",
                      "oidc_metadata", "mfa_surface", "authorization_surface",
                      "role", "permission", "resource_access", "access_control"):
    assert expected_kind in kinds, expected_kind

# Every observation evidence row is DERIVED_FROM its run's artifact row.
rel_ok = True
count_obs = 0
for r in pipeline_runs:
    artifact = r.evidence_ids[0]
    for ev_id in r.evidence_ids[1:]:
        rels = app.evidence_store.get_relationships(ev_id)
        ok = any(
            x.relation_type == EvidenceRelation.DERIVED_FROM
            and str(x.target_id) == str(artifact)
            for x in rels
        )
        rel_ok = rel_ok and ok
        count_obs += 1
assert rel_ok and count_obs >= 11
print(f"DERIVED_FROM links: {count_obs} observations across {len(pipeline_runs)} artifacts")

# World model: authenticated web application materialized from observations.
wm = engine.world_model
app_entity = wm.find_entity(MID, EntityType.APPLICATION, "web.example.com")
assert app_entity is not None, "APPLICATION entity must exist"
auth_entity = wm.find_entity(
    MID, EntityType.AUTHENTICATION, "session_cookie",
    namespace="web.example.com",
)
assert auth_entity is not None, "AUTHENTICATION entity must exist"
ep_entity = wm.find_entity(MID, EntityType.ENDPOINT, "https://web.example.com/")
assert ep_entity is not None, "ENDPOINT entity must exist"
print("APPLICATION:", app_entity.name, "| AUTHENTICATION:", auth_entity.name,
      "| ENDPOINT:", ep_entity.name)

# Identity -> role -> permission -> resource chain (namespaced by host).
alice = wm.find_entity(MID, EntityType.IDENTITY, "alice", namespace="web.example.com")
role_entity = wm.find_entity(MID, EntityType.ROLE, "editor", namespace="web.example.com")
perm_entity = wm.find_entity(MID, EntityType.PERMISSION, "create::reports",
                             namespace="web.example.com")
res_entity = wm.find_entity(MID, EntityType.RESOURCE, "reports",
                            namespace="web.example.com")
assert alice is not None and role_entity is not None
assert perm_entity is not None and res_entity is not None
print("IDENTITY:", alice.name, "| ROLE:", role_entity.name,
      "| PERMISSION:", perm_entity.name, "| RESOURCE:", res_entity.name)

rels = wm.list_relationships(RelationshipQuery(mission_id=MID, limit=1000))
allowed_types = {"has_role", "has_permission", "applies_to",
                 "requires", "contains", "runs"}
rel_types = {getattr(r.relationship_type, "value", r.relationship_type) for r in rels}
assert rel_types <= allowed_types, f"Unexpected relationship types: {rel_types}"
print("Relationship types:", ", ".join(sorted(rel_types)))
print("No attack-graph relationship types (EXPLOITS/CAN_COMPROMISE/LEADS_TO/ENABLES): PASS")

# Assertions bound to the host APPLICATION; exercised access validated on IDENTITY.
app_assertions = {a.property_key: a for a in wm.list_assertions(str(app_entity.id))}
alice_assertions = wm.list_assertions(str(alice.id))
assert "auth_scheme.session_cookie" in app_assertions
assert "session.session" in app_assertions
assert "mfa" in app_assertions and "authorization.model" in app_assertions
assert len(alice_assertions) > 0
app_statuses = {a.epistemic_status.value for a in wm.list_assertions(str(app_entity.id))}
alice_statuses = {a.epistemic_status.value for a in wm.list_assertions(str(alice.id))}
# exercised resource_access / access_control assertions are VALIDATED on the
# identity; observation-derived analysis stays OBSERVED on the application
assert "validated" in alice_statuses, alice_statuses
assert "observed" in app_statuses, app_statuses
print(f"APPLICATION-bound assertions: {len(app_assertions)} ({app_statuses}) | "
      f"IDENTITY alice assertions: {len(alice_assertions)} ({alice_statuses})")

# Access-control comparison outcome for explicit test identities.
cmp_obs = {o.identity: o for o in r_cmp.observations}
assert set(cmp_obs) == {"alice", "bob"}, list(cmp_obs)
assert cmp_obs["alice"].consistent is True
print(f"access_control_comparison -> alice consistent={cmp_obs['alice'].consistent} "
      f"({cmp_obs['alice'].resource}:{cmp_obs['alice'].access.value})")


---


In [ ]:
# Mission isolation: work under a second mission is fully disjoint.
MID2 = "mission_phase7_auth_other"
scope2 = TargetScope(
    mission_id=MID2,
    allowed_targets=[_target("web.example.com")],
    max_risk_level=RiskLevel.HIGH,
)
req2 = AuthRequest(mission_id=MID2, scope=scope2, test_identities=["alice"])
res2 = engine.observe_authentication_surface(req2, "web.example.com")
other_ids = {str(x) for x in res2.evidence_ids}
assert other_ids.isdisjoint({str(x) for x in r_surface.evidence_ids})
assert app.evidence_store.count(MID2) == len(res2.evidence_ids)
assert engine.world_model.count_entities(MID2) >= 1
print("Mission isolation: second mission produced its own evidence/world rows: PASS")

# Redaction: session values, tokens and credential values are never plaintext.
import hashlib as _hashlib

assert "mock-session-web" not in r_session.raw_output, "Leaked session value in raw output"
assert r_session.observations[0].value_hashed == _hashlib.sha256(b"mock-session-web").hexdigest()
for secret in ("mock-bearer", "mock-password"):
    assert secret not in r_access.raw_output, f"Leaked {secret} in raw output"
assert r_perm.observations[0].credential_value == "REDACTED"
assert r_access.observations[0].credential_value == "REDACTED"
print("Redaction at the boundary (one-way digest + literal REDACTED): PASS")


---


In [ ]:
from blackforge.core.errors import AuthorizationError, AuthExecutionError

# 1) Target outside the scope is denied BEFORE any transport runs.
denied_out = False
try:
    engine.observe_authentication_surface(req, "scanme.example.org")
except AuthorizationError:
    denied_out = True
assert denied_out
print("Out-of-scope target denied before transport execution: PASS")

# 2) Unknown / non-typed capability is rejected.
unknown_rejected = False
try:
    engine.run(req, "auth.not_real", "web.example.com")
except AuthExecutionError:
    unknown_rejected = True
assert unknown_rejected
print("Unknown capability rejected (no generic execution surface): PASS")

# 3) Access-validation without explicit test identities is rejected (no guessing).
no_ids_req = AuthRequest(mission_id=MID, scope=scope, test_identities=[])
missing = False
try:
    engine.observe_resource_access(no_ids_req, "web.example.com")
except AuthExecutionError as e:
    missing = "requires explicit" in str(e)
assert missing
print("Missing test identities rejected (no default guessing): PASS")

# 4) Failure states: NO_EVIDENCE / RATE_LIMITED / REQUEST_FAILED / LIMITED.
res = engine.observe_authentication_surface(req, "mail.example.com")
assert res.status.value == "no_evidence" and not res.observations and res.warnings
res = engine.observe_authentication_surface(req, "throttled.example.com")
assert res.status.value == "rate_limited" and "rate limited" in (res.error or "")
res = engine.observe_authentication_surface(req, "unreachable.example.com")
assert res.status.value == "request_failed" and "connection refused" in (res.error or "")
limited_req = AuthRequest(
    mission_id=MID, scope=scope, test_identities=["alice"], max_observations=1,
)
res = engine.observe_roles(limited_req, "web.example.com")
assert res.status.value == "limited" and len(res.observations) == 1
assert any("limit" in w for w in res.warnings)
print("Failure states (NO_EVIDENCE, RATE_LIMITED, REQUEST_FAILED, LIMITED): PASS")


---


In [ ]:
from blackforge.evidence.repository import SQLiteEvidenceRepository
from blackforge.evidence.store import EvidenceStore
from blackforge.world_model.repository import SQLiteWorldRepository
from blackforge.world_model.store import WorldModelStore

# Fresh connections over the same SQLite files prove restart persistence.
fresh_ev = EvidenceStore(SQLiteEvidenceRepository(str(DBROOT / "evidence.db")))
fresh_wm = WorldModelStore(SQLiteWorldRepository(str(DBROOT / "world_model.db")))

persisted_ev = fresh_ev.count(MID) == app.evidence_store.count(MID)
persisted_wm = fresh_wm.count_entities(MID) == engine.world_model.count_entities(MID)
api_entity = fresh_wm.find_entity(MID, EntityType.APPLICATION, "web.example.com")
alice_entity = fresh_wm.find_entity(
    MID, EntityType.IDENTITY, "alice", namespace="web.example.com",
)
persisted_api = api_entity is not None
persisted_alice = alice_entity is not None
assert persisted_ev and persisted_wm and persisted_api and persisted_alice
rel_count = len(fresh_wm.list_relationships(RelationshipQuery(mission_id=MID, limit=1000)))
assert rel_count > 0
assert fresh_ev.count(MID) > 0
PERSIST_OK = persisted_ev and persisted_wm and persisted_api and persisted_alice

for store in (fresh_ev, fresh_wm):
    store.close()
try:
    app.evidence_store.close()
except Exception:
    pass
try:
    app.world_model.close()
except Exception:
    pass
print("Restart persistence (fresh connections on same DB files): PASS")
print("Backends closed. Validation summary below.")


---


In [ ]:
results = {}
phase_checks = {
    "repository_integrity": (REPO_DIR / "blackforge" / "auth" / "engine.py").exists(),
    "phase7_modules": bool(
        (REPO_DIR / "blackforge" / "auth" / "capabilities.py").exists()
        and (REPO_DIR / "blackforge" / "auth" / "evidence.py").exists()
        and (REPO_DIR / "blackforge" / "auth" / "materializer.py").exists()
        and (REPO_DIR / "blackforge" / "auth" / "redaction.py").exists()
    ),
    "imports": len(_import_failures) == 0,
    "bootstrap_auth_ready": BOOTSTRAP_OK,
    "no_generic_executor": CAPS_OK,
    "capability_surface": CAPS_OK,
    "pipeline_evidence": rel_ok,
    "world_materialized": app_entity is not None and auth_entity is not None and ep_entity is not None,
    "identity_chain": alice is not None and role_entity is not None and perm_entity is not None and res_entity is not None,
    "assertions_bound": len(app_assertions) > 0 and len(alice_assertions) > 0 and "validated" in alice_statuses,
    "no_attack_graph": not (rel_types & {"exploits", "can_compromise", "leads_to", "enables"}),
    "scope_authorization": denied_out,
    "unknown_capability_rejected": unknown_rejected,
    "test_identities_required": missing,
    "mission_isolation": bool(other_ids.isdisjoint({str(x) for x in r_surface.evidence_ids})),
    "restart_persistence": PERSIST_OK,
}

# The pytest cell aborts the run on failure, so reaching this cell proves it passed.
pytest_passed = True
install_ok = len(_import_failures) == 0

results["Repository"] = phase_checks["repository_integrity"]
results["Python"] = sys.version_info >= (3, 10)
results["Hardware"] = True  # CPU fallback always works; this notebook needs no GPU
results["Installation"] = install_ok
results["Imports"] = install_ok
results["Automated tests"] = pytest_passed
results["Bootstrap"] = phase_checks["bootstrap_auth_ready"]
results["Phase-specific tests"] = all(phase_checks.values())
results["Security checks"] = (
    phase_checks["scope_authorization"]
    and phase_checks["unknown_capability_rejected"]
    and phase_checks["test_identities_required"]
    and phase_checks["no_attack_graph"]
)

print()
print("=" * 60)
print("PHASE 7 COLAB VALIDATION SUMMARY")
print("=" * 60)
for name, ok in results.items():
    symbol = "PASS" if ok else "FAIL"
    print(f"  [{symbol}] {name}")

_all_ok = all(results.values()) and all(phase_checks.values())
assert _all_ok, "One or more validation checks failed"

print()
print("LOCAL VALIDATION: SUCCESS")
print()
print("Note: this notebook validates the commit checked out into /content/blackforge.")
